# Windows-native Wolof–French alignment with IBM Model 1

This notebook replaces the **lexicon-building role** of Eflomal with NLTK's pure-Python IBM Model 1. It learns word alignments from the same French–Wolof parallel corpus and exports the same structure used by the existing pipeline:

```python
{"wolof_word": {"french_word": probability}}
```

The output is saved as `artifacts/ibm1_lexicon.pkl`. It does **not** overwrite the active Eflomal artifact `artifacts/lexicon.pkl`.

In [1]:
from pathlib import Path
import pickle
import sys
import time

import pandas as pd

# Work whether Jupyter starts in the project root or in notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.alignment.ibm1_lexicon import (
    save_lexicon,
    top_translations,
    train_lexicon,
    word_tokens,
)
from src.config import ARTIFACTS_DIR, LEXICON_PATH, TRAIN_PARQUET_PATH

IBM1_LEXICON_PATH = ARTIFACTS_DIR / "ibm1_lexicon.pkl"
print(f"Project root: {PROJECT_ROOT}")
print(f"Parallel corpus: {TRAIN_PARQUET_PATH}")
print(f"IBM1 output: {IBM1_LEXICON_PATH}")

Project root: C:\Users\thioy\Desktop\School\PFE
Parallel corpus: C:\Users\thioy\Desktop\School\PFE\data\translation data\train-00000-of-00001.parquet
IBM1 output: C:\Users\thioy\Desktop\School\PFE\artifacts\ibm1_lexicon.pkl


## 1. Trial configuration

Start with 2,000 randomly selected sentence pairs. After inspecting the output, set `MAX_SENTENCES = None` and increase `ITERATIONS` to 10 for the full artifact.

In [18]:
MAX_SENTENCES = None   # Set to None for all 17,777 pairs
ITERATIONS = 10          # Use 10 for the final full-corpus run
MIN_COUNT = 3           # Use 3 for a stricter final lexicon
MIN_PROB = 0.10
RANDOM_SEED = 2026

## 2. Load and inspect the parallel corpus

In [19]:
parallel_df = pd.read_parquet(TRAIN_PARQUET_PATH)
parallel_df = parallel_df.dropna(subset=["wolof", "french"]).copy()

if MAX_SENTENCES is not None and MAX_SENTENCES < len(parallel_df):
    training_df = parallel_df.sample(
        n=MAX_SENTENCES, random_state=RANDOM_SEED
    ).reset_index(drop=True)
else:
    training_df = parallel_df.reset_index(drop=True)

print(f"Available pairs: {len(parallel_df):,}")
print(f"Pairs in this run: {len(training_df):,}")
display(training_df[["wolof", "french"]].head(10))

Available pairs: 17,777
Pairs in this run: 17,777


,wolof,french
0,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...,Cette lettre doit être légalisée par le Minist...
1,"""Amuñu woon benn jot ngir rawal sunu bopp.\n",« On n’a pas eu le temps de se sauver.\n
2,"Ekost sa dëkk la te bëggna nu nga toog fi.""\n",L’Écosse est tout à fait votre maison et nous ...
3,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...,Jotnanews: L’Espagne injecte 84 milliards FCFA...
4,ñaata at nga am,quel âge as-tu
5,"Abu Usmaan Si, Mamadu Yoro Jàllo ak Usmaan Nja...","Abou Ousmane Sy, Mamadou Yoro Diallo et Ousman..."
6,Ni ñu ko tàmm a waxe fii : boo xamatul foo jëm...,Et comme l’on a coutume de le dire dans nos co...
7,"Zambie réew la mu bokk ci ONU, Union Africaine...","La Zambie est membre des Nations Unies, de l'U..."
8,tooy,humide
9,"Ay teemeeri nit faatu ca donu bu Nias, ci tefe...",Quelques centaines de personnes sont décédées ...


## 3. Train IBM Model 1 and build the lexicon

NLTK uses `AlignedSent.words` as the target and `AlignedSent.mots` as the source. The reusable helper deliberately stores French as the target and Wolof as the source, producing Wolof → French probabilities.

In [20]:
started = time.perf_counter()
ibm1_lexicon, ibm1_model, aligned_bitext = train_lexicon(
    training_df,
    iterations=ITERATIONS,
    min_count=MIN_COUNT,
    min_prob=MIN_PROB,
    wolof_col="wolof",
    french_col="french",
)
elapsed = time.perf_counter() - started

translation_count = sum(len(values) for values in ibm1_lexicon.values())
print(f"Training time: {elapsed:.1f} seconds")
print(f"Aligned sentence pairs: {len(aligned_bitext):,}")
print(f"Wolof entries: {len(ibm1_lexicon):,}")
print(f"Retained Wolof–French links: {translation_count:,}")

Training time: 81.4 seconds
Aligned sentence pairs: 17,777
Wolof entries: 4,914
Retained Wolof–French links: 8,704


## 4. Inspect learned translations

Add frequent Wolof words from your corpus to `WORDS_TO_INSPECT`. Empty results usually mean the trial sample or `MIN_COUNT` is too small/strict.

In [21]:
WORDS_TO_INSPECT = ["ak", "ci", "joxe", "laaj", "nit", "kër"]

inspection_rows = []
for wolof_word in WORDS_TO_INSPECT:
    translations = top_translations(ibm1_lexicon, wolof_word, limit=5)
    inspection_rows.append(
        {
            "wolof": wolof_word,
            "top_french_translations": translations,
        }
    )
pd.DataFrame(inspection_rows)

,wolof,top_french_translations
0,ak,"[(et, 0.8275058275058275), (avec, 0.1717171717..."
1,ci,"[(dans, 0.43933649289099524), (en, 0.318483412..."
2,joxe,"[(en, 0.4885057471264368), (par, 0.17241379310..."
3,laaj,"[(questions, 0.38372093023255816), (question, ..."
4,nit,"[(personnes, 0.4511784511784512), (gens, 0.410..."
5,kër,"[(la, 0.391304347826087), (maison, 0.380434782..."


## 5. Inspect sentence-level alignments

The stored pairs use `(French index, Wolof index)`. The cell below converts them into readable Wolof–French word pairs.

In [22]:
def readable_alignment(aligned_sentence):
    rows = []
    valid_links = [link for link in aligned_sentence.alignment if None not in link]
    for french_index, wolof_index in sorted(valid_links):
        rows.append(
            {
                "wolof_index": wolof_index,
                "wolof_word": aligned_sentence.mots[wolof_index],
                "french_index": french_index,
                "french_word": aligned_sentence.words[french_index],
            }
        )
    return pd.DataFrame(rows)

EXAMPLE_INDEX = 0
example = aligned_bitext[EXAMPLE_INDEX]
print("Wolof:", " ".join(example.mots))
print("French:", " ".join(example.words))
display(readable_alignment(example))

Wolof: bataaxal bii jëwriñu lu ajju ci mbiru bitim réew bu ekuwatër moo ko wara wóoral te dafa wara mengoo ak yenni càkuteef yi
French: cette lettre doit être légalisée par le ministère des affaires étrangères équatorien et respecter certaines conditions


,wolof_index,wolof_word,french_index,french_word
0,1,bii,0,cette
1,0,bataaxal,1,lettre
2,14,wóoral,2,doit
3,17,wara,3,être
4,21,càkuteef,4,légalisée
5,6,mbiru,5,par
6,11,moo,6,le
7,2,jëwriñu,7,ministère
8,22,yi,8,des
9,21,càkuteef,9,affaires


## 6. Compare with the existing Eflomal lexicon

Top-translation agreement is a diagnostic, not proof that either aligner is correct. Manually review disagreements.

In [23]:
if LEXICON_PATH.exists():
    with open(LEXICON_PATH, "rb") as handle:
        eflomal_lexicon = pickle.load(handle)

    common_words = sorted(set(ibm1_lexicon) & set(eflomal_lexicon))
    agreements = []
    disagreements = []
    for wolof_word in common_words:
        ibm_top = max(ibm1_lexicon[wolof_word], key=ibm1_lexicon[wolof_word].get)
        eflomal_top = max(eflomal_lexicon[wolof_word], key=eflomal_lexicon[wolof_word].get)
        if ibm_top == eflomal_top:
            agreements.append(wolof_word)
        else:
            disagreements.append(
                {
                    "wolof": wolof_word,
                    "ibm1_top": ibm_top,
                    "eflomal_top": eflomal_top,
                }
            )

    agreement_rate = len(agreements) / len(common_words) if common_words else 0
    print(f"Eflomal Wolof entries: {len(eflomal_lexicon):,}")
    print(f"Common Wolof entries: {len(common_words):,}")
    print(f"Top-translation agreement: {agreement_rate:.1%}")
    display(pd.DataFrame(disagreements).head(30))
else:
    print(f"No Eflomal artifact found at {LEXICON_PATH}")

Eflomal Wolof entries: 1,989
Common Wolof entries: 1,965
Top-translation agreement: 78.4%


,wolof,ibm1_top,eflomal_top
0,a,qui,de
1,aada,culturel,culture
2,aali,ali,aly
3,aguero,a,aguero
4,aki,et,les
5,alaaji,el,hadji
6,almaañ,en,l
7,amna,a,y
8,amul,pas,sans
9,amuñu,pas,n


## 7. Test the paired-sentence substitution constraint

This reproduces the important downstream rule from the Eflomal notebook: a French translation may be selected only when it appears in the paired French sentence.

In [24]:
def substitution_candidates(wolof_text, french_text, lexicon):
    wolof_tokens = word_tokens(wolof_text)
    french_tokens = word_tokens(french_text)
    french_set = set(french_tokens)
    candidates = []
    for wolof_index, wolof_word in enumerate(wolof_tokens):
        constrained = {
            french_word: probability
            for french_word, probability in lexicon.get(wolof_word, {}).items()
            if french_word in french_set
        }
        if constrained:
            french_word = max(constrained, key=constrained.get)
            candidates.append(
                (wolof_index, wolof_word, french_word, constrained[french_word])
            )
    return sorted(candidates, key=lambda item: item[3], reverse=True)

row = training_df.iloc[0]
print("Wolof:", row["wolof"])
print("French:", row["french"])
pd.DataFrame(
    substitution_candidates(row["wolof"], row["french"], ibm1_lexicon),
    columns=["wolof_index", "wolof_word", "french_word", "probability"],
)

Wolof: Bataaxal bii jëwriñu lu ajju ci mbiru bitim réew bu Ekuwatër moo ko wara wóoral, te dafa wara mengoo ak yenni càkuteef yi.

French: Cette lettre doit être légalisée par le Ministère des affaires étrangères équatorien et respecter certaines conditions.



,wolof_index,wolof_word,french_word,probability
0,15,te,et,0.995998
1,19,ak,et,0.827506
2,13,wara,être,0.452830
3,17,wara,être,0.452830
4,0,bataaxal,lettre,0.431034
5,1,bii,cette,0.377778
6,7,bitim,étrangères,0.222222
7,22,yi,des,0.220342
8,11,moo,le,0.216931
9,12,ko,par,0.188571


## 8. Save the separate IBM1 artifact

This remains separate from `lexicon.pkl`. The current pipeline will continue using Eflomal until you deliberately compare and switch artifacts.

In [25]:
saved_path = save_lexicon(ibm1_lexicon, IBM1_LEXICON_PATH)
print(f"Saved IBM Model 1 lexicon to: {saved_path}")
print(f"Active Eflomal lexicon remains: {LEXICON_PATH}")

Saved IBM Model 1 lexicon to: C:\Users\thioy\Desktop\School\PFE\artifacts\ibm1_lexicon.pkl
Active Eflomal lexicon remains: C:\Users\thioy\Desktop\School\PFE\artifacts\lexicon.pkl


## Final run and validation

For the final artifact, rerun from the configuration cell with:

```python
MAX_SENTENCES = None
ITERATIONS = 10
MIN_COUNT = 3
MIN_PROB = 0.10
```

Before replacing Eflomal anywhere, manually inspect at least 100 common words and sentence alignments. Record coverage, top-translation accuracy, runtime, and representative disagreements for the report.